# 06 — Ownership vs Rental TCO Comparison

This notebook compares two fundamentally different approaches to GPU cluster costing:

| | **DC Ownership (Lifecycle)** | **Cloud Rental (SemiAnalysis-style)** |
|---|---|---|
| **Scope** | Build + provision + operate a datacenter over 15 years | Rent GPUs from a cloud provider on a monthly contract |
| **Cost structure** | CapEx-heavy (servers, facility, cooling) + OpEx (power, maintenance) | OpEx-only (GPU $/hr + storage + networking + support) |
| **Time horizon** | Multi-year lifecycle with hardware refresh cycles | Monthly billing, typically 1–3 year contracts |
| **Goodput** | Implicit in utilization factor | Explicit formula: checkpoint, hot-spare, or fault-tolerant |
| **Key trade-off** | CapEx amortization vs fleet aging | Effective $/GPU-hr vs list price |

We run both models on a **1024-GPU B200 cluster** and compare:
1. Cost breakdown structure
2. Monthly / annual cost equivalence
3. Sensitivity to key parameters (GPU price, goodput mode, contract term)
4. Break-even analysis: when does owning beat renting?

In [ ]:
import sys
sys.path.insert(0, '..')

import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from dc_tco.config import load_config, GoodputMode
from dc_tco.simulation import run_simulation
from dc_tco.rental import (
    compute_rental_monthly,
    compute_rental_tco,
    compute_goodput,
    goodput_checkpoint_cold,
    goodput_checkpoint_hot,
    goodput_fault_tolerant,
)

MONTHS_PER_YEAR = 12

plt.rcParams.update({
    'figure.figsize': (12, 5),
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

cfg = load_config('../configs/default.yaml')
print(f"Loaded config: {cfg.simulation.total_years}-year lifecycle, "
      f"{cfg.rental.num_gpus} GPUs rental cluster")

## 1. Ownership TCO — Lifecycle Simulation

Run the 15-year DC lifecycle simulation and extract the CapEx/OpEx breakdown.

In [ ]:
own_result = run_simulation(cfg)

# Annual ownership costs
own_years = sorted(own_result.tco_by_year.keys())
own_annual = []
for y in own_years:
    tb = own_result.tco_by_year[y]
    own_annual.append({
        'year': y,
        'capex_server': tb.capex_server,
        'capex_rack': tb.capex_rack,
        'capex_power': tb.capex_power_provisioning,
        'capex_cooling': tb.capex_cooling_provisioning,
        'capex_facility': tb.capex_facility,
        'capex_network': tb.capex_network,
        'opex_energy': tb.opex_energy,
        'opex_cooling': tb.opex_cooling,
        'opex_maintenance': tb.opex_maintenance,
        'opex_network': tb.opex_network,
        'total_capex': tb.total_capex,
        'total_opex': tb.total_opex,
        'total': tb.total,
    })

own_df = pd.DataFrame(own_annual)

print(f"Total ownership TCO (15yr): ${own_result.total_tco/1e9:.2f}B")
print(f"  CapEx: ${own_result.tco_breakdown.total_capex/1e9:.2f}B")
print(f"  OpEx:  ${own_result.tco_breakdown.total_opex/1e9:.2f}B")
print(f"  CapEx/Total: {own_result.tco_breakdown.total_capex/own_result.total_tco:.1%}")

## 2. Rental TCO — SemiAnalysis 8-Component Model

Compute the monthly rental TCO and break it down by the 8 components.

In [ ]:
rental_result = compute_rental_tco(cfg)
m = rental_result.monthly
gm = rental_result.goodput_metrics

# Monthly breakdown table
rental_components = {
    'GPU Compute': m.gpu,
    'Storage': m.storage,
    'Networking': m.networking,
    'Control Plane': m.control_plane,
    'Support': m.support,
    'Goodput Loss': m.goodput,
    'Setup (amortized)': m.setup,
    'Debugging': m.debugging,
}

rental_df = pd.DataFrame([
    {'Component': k, 'Monthly ($)': v, '% of Total': v / m.total * 100}
    for k, v in rental_components.items()
])
rental_df.loc[len(rental_df)] = ['TOTAL', m.total, 100.0]

print(f"Rental: {cfg.rental.num_gpus} GPUs @ ${cfg.rental.gpu_cost_per_hour}/GPU-hr")
print(f"Contract: {cfg.rental.contract_months} months")
print(f"Goodput mode: {cfg.rental.goodput.mode}")
print(f"Goodput utilization: {gm.utilization:.1%}")
print(f"Effective $/GPU-hr: ${rental_result.cost_per_gpu_hour_effective:.2f}")
print()
print(rental_df.to_string(index=False, float_format='${:,.0f}'.format))

In [ ]:
# Pie chart: rental cost breakdown
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Rental breakdown
labels = list(rental_components.keys())
values = list(rental_components.values())
colors = plt.cm.Set3(np.linspace(0, 1, len(labels)))
wedges, texts, autotexts = ax1.pie(
    values, labels=labels, autopct='%1.1f%%',
    colors=colors, startangle=90, pctdistance=0.8,
)
ax1.set_title(f'Rental TCO — Monthly ${m.total:,.0f}\n'
              f'({cfg.rental.num_gpus} GPUs, {cfg.rental.goodput.mode})')

# Ownership breakdown (annualized)
own_capex_components = {
    'Server': own_result.tco_breakdown.capex_server,
    'Rack': own_result.tco_breakdown.capex_rack,
    'Power Prov.': own_result.tco_breakdown.capex_power_provisioning,
    'Cooling Prov.': own_result.tco_breakdown.capex_cooling_provisioning,
    'Facility': own_result.tco_breakdown.capex_facility,
    'Network (CapEx)': own_result.tco_breakdown.capex_network,
}
own_opex_components = {
    'Energy': own_result.tco_breakdown.opex_energy,
    'Cooling (OpEx)': own_result.tco_breakdown.opex_cooling,
    'Maintenance': own_result.tco_breakdown.opex_maintenance,
    'Network (OpEx)': own_result.tco_breakdown.opex_network,
}
own_all = {**{f'CapEx: {k}': v for k, v in own_capex_components.items()},
           **{f'OpEx: {k}': v for k, v in own_opex_components.items()}}
own_labels = [k for k, v in own_all.items() if v > 0]
own_values = [v for v in own_all.values() if v > 0]
own_colors = plt.cm.Paired(np.linspace(0, 1, len(own_labels)))
wedges2, texts2, autotexts2 = ax2.pie(
    own_values, labels=own_labels, autopct='%1.1f%%',
    colors=own_colors, startangle=90, pctdistance=0.8,
)
ax2.set_title(f'Ownership TCO — {cfg.simulation.total_years}yr total '
              f'${own_result.total_tco/1e9:.1f}B')

plt.tight_layout()
plt.show()

## 3. Head-to-Head: Annualized Cost Comparison

Normalize both models to an **annual cost per GPU** for direct comparison.

In [ ]:
# Rental: annualized per GPU
rental_annual_per_gpu = m.total * MONTHS_PER_YEAR / cfg.rental.num_gpus

# Ownership: we need to estimate per-GPU cost from the simulation
# Use average annual TCO / average GPU count across the lifecycle
avg_annual_own = own_result.total_tco / cfg.simulation.total_years
avg_servers = np.mean([s.total_servers for s in own_result.quarterly_states])
gpus_per_server = cfg.hardware.known_gpus[0].get('gpus_per_server', 8) if cfg.hardware.known_gpus else 8
avg_gpus = avg_servers * gpus_per_server
own_annual_per_gpu = avg_annual_own / max(avg_gpus, 1)

# Effective $/GPU-hr
rental_eff_per_hr = rental_result.cost_per_gpu_hour_effective
own_eff_per_hr = own_annual_per_gpu / (24 * 365)

comparison = pd.DataFrame([
    {'Metric': 'Annual cost per GPU',
     'Ownership': f'${own_annual_per_gpu:,.0f}',
     'Rental': f'${rental_annual_per_gpu:,.0f}'},
    {'Metric': 'Effective $/GPU-hr',
     'Ownership': f'${own_eff_per_hr:.2f}',
     'Rental': f'${rental_eff_per_hr:.2f}'},
    {'Metric': 'Monthly total (cluster)',
     'Ownership': f'${avg_annual_own/12:,.0f}',
     'Rental': f'${m.total:,.0f}'},
    {'Metric': 'GPU % of total',
     'Ownership': f'{own_result.tco_breakdown.capex_server/own_result.total_tco:.1%}',
     'Rental': f'{m.gpu/m.total:.1%}'},
])

print(comparison.to_string(index=False))

In [ ]:
# Bar chart: annual per-GPU cost breakdown
fig, ax = plt.subplots(figsize=(10, 6))

# Ownership categories (annualized per GPU)
own_cats = {
    'Server CapEx': own_result.tco_breakdown.capex_server,
    'Infra CapEx': (own_result.tco_breakdown.capex_rack
                    + own_result.tco_breakdown.capex_power_provisioning
                    + own_result.tco_breakdown.capex_cooling_provisioning
                    + own_result.tco_breakdown.capex_facility
                    + own_result.tco_breakdown.capex_network),
    'Energy': own_result.tco_breakdown.opex_energy,
    'Cooling & Maint.': (own_result.tco_breakdown.opex_cooling
                         + own_result.tco_breakdown.opex_maintenance
                         + own_result.tco_breakdown.opex_network),
}
own_per_gpu = {k: v / cfg.simulation.total_years / max(avg_gpus, 1)
               for k, v in own_cats.items()}

# Rental categories (annualized per GPU)
rental_cats = {
    'GPU Compute': m.gpu * 12 / cfg.rental.num_gpus,
    'Storage + Network': (m.storage + m.networking) * 12 / cfg.rental.num_gpus,
    'Support + Control': (m.support + m.control_plane) * 12 / cfg.rental.num_gpus,
    'Goodput + Debug': (m.goodput + m.debugging + m.setup) * 12 / cfg.rental.num_gpus,
}

x = np.arange(2)
width = 0.18

# Stack bars for each model
own_vals = list(own_per_gpu.values())
rental_vals = list(rental_cats.values())

# Ownership stacked bar
own_colors = ['#2196F3', '#64B5F6', '#FF9800', '#FFB74D']
bottom = 0
for i, (label, val) in enumerate(own_per_gpu.items()):
    ax.bar(0, val, width=0.5, bottom=bottom, color=own_colors[i], label=f'Own: {label}')
    bottom += val

# Rental stacked bar
rental_colors = ['#4CAF50', '#81C784', '#9C27B0', '#CE93D8']
bottom = 0
for i, (label, val) in enumerate(rental_cats.items()):
    ax.bar(1, val, width=0.5, bottom=bottom, color=rental_colors[i], label=f'Rent: {label}')
    bottom += val

ax.set_xticks([0, 1])
ax.set_xticklabels(['DC Ownership\n(annualized)', 'Cloud Rental\n(annual)'])
ax.set_ylabel('Cost per GPU per Year ($)')
ax.set_title('Annual Cost per GPU — Ownership vs Rental')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Goodput Sensitivity — Three Failure Modes

Compare the three SemiAnalysis goodput formulas and their impact on effective cost.

In [ ]:
goodput_modes = [
    (GoodputMode.CHECKPOINT_COLD, 'Checkpoint (Cold)', goodput_checkpoint_cold),
    (GoodputMode.CHECKPOINT_HOT, 'Checkpoint (Hot Spare)', goodput_checkpoint_hot),
    (GoodputMode.FAULT_TOLERANT, 'Fault Tolerant (TorchFT)', goodput_fault_tolerant),
]

failure_rates = np.arange(0, 25, 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for mode_key, mode_label, mode_fn in goodput_modes:
    utils = []
    expenses = []
    for nf in failure_rates:
        c = copy.deepcopy(cfg)
        c.rental.goodput.mode = mode_key
        c.rental.goodput.num_failures_per_month = float(nf)
        gm = mode_fn(c)
        utils.append(gm.utilization * 100)
        expenses.append(gm.goodput_expense)
    ax1.plot(failure_rates, utils, label=mode_label, linewidth=2)
    ax2.plot(failure_rates, expenses, label=mode_label, linewidth=2)

ax1.set_xlabel('Failures per Month')
ax1.set_ylabel('Goodput Utilization (%)')
ax1.set_title('Goodput Utilization vs Failure Rate')
ax1.legend()
ax1.set_ylim(50, 105)

ax2.set_xlabel('Failures per Month')
ax2.set_ylabel('Wasted Spend ($/month)')
ax2.set_title('Goodput Expense vs Failure Rate')
ax2.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
ax2.legend()

plt.suptitle(f'{cfg.rental.num_gpus} GPUs @ ${cfg.rental.gpu_cost_per_hour}/GPU-hr',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 5. Effective $/GPU-hr by Goodput Mode

What you *actually* pay per useful GPU-hour after accounting for all costs.

In [ ]:
rows = []
for mode_key, mode_label, _ in goodput_modes:
    c = copy.deepcopy(cfg)
    c.rental.goodput.mode = mode_key
    result = compute_rental_tco(c)
    rows.append({
        'Mode': mode_label,
        'Monthly Total': f'${result.monthly.total:,.0f}',
        'Goodput Util.': f'{result.goodput_metrics.utilization:.1%}',
        'Eff. $/GPU-hr': f'${result.cost_per_gpu_hour_effective:.2f}',
        'Contract Total': f'${result.contract_total.total:,.0f}',
    })

# Add ownership equivalent
rows.append({
    'Mode': 'DC Ownership (equiv.)',
    'Monthly Total': f'${avg_annual_own/12:,.0f}',
    'Goodput Util.': f'{cfg.tco.utilization:.1%}',
    'Eff. $/GPU-hr': f'${own_eff_per_hr:.2f}',
    'Contract Total': f'${own_result.total_tco:,.0f}',
})

print(pd.DataFrame(rows).to_string(index=False))

## 6. Break-Even Analysis: Own vs Rent

At what GPU rental price does owning become cheaper (and vice versa)?
We vary the GPU $/hr and compute the crossover point.

In [ ]:
gpu_prices = np.arange(0.50, 6.01, 0.10)
rental_annual_costs = []

for price in gpu_prices:
    c = copy.deepcopy(cfg)
    c.rental.gpu_cost_per_hour = float(price)
    monthly = compute_rental_monthly(c)
    rental_annual_costs.append(monthly.total * 12)

rental_annual_costs = np.array(rental_annual_costs)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(gpu_prices, rental_annual_costs / 1e6,
        label='Cloud Rental (annual)', color='#4CAF50', linewidth=2)
ax.axhline(avg_annual_own / 1e6, color='#2196F3', linewidth=2,
           linestyle='--', label='DC Ownership (annual avg)')

# Find crossover
cross_idx = np.argmin(np.abs(rental_annual_costs - avg_annual_own))
cross_price = gpu_prices[cross_idx]
ax.axvline(cross_price, color='red', linewidth=1, linestyle=':',
           label=f'Break-even: ${cross_price:.2f}/GPU-hr')
ax.scatter([cross_price], [avg_annual_own / 1e6], color='red', zorder=5, s=80)

ax.set_xlabel('GPU Rental Price ($/GPU-hr)')
ax.set_ylabel('Annual Cluster Cost ($M)')
ax.set_title(f'Break-Even: Ownership vs Rental ({cfg.rental.num_gpus} GPUs)')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}M'))
ax.legend()
ax.fill_between(gpu_prices[:cross_idx+1],
                rental_annual_costs[:cross_idx+1] / 1e6,
                avg_annual_own / 1e6,
                alpha=0.15, color='green', label='_Rental cheaper')
ax.fill_between(gpu_prices[cross_idx:],
                rental_annual_costs[cross_idx:] / 1e6,
                avg_annual_own / 1e6,
                alpha=0.15, color='blue', label='_Own cheaper')
ax.annotate('Rental\ncheaper', xy=(cross_price - 0.8, avg_annual_own / 1e6 * 0.7),
            fontsize=11, color='green', ha='center')
ax.annotate('Owning\ncheaper', xy=(cross_price + 0.8, avg_annual_own / 1e6 * 1.3),
            fontsize=11, color='blue', ha='center')
plt.tight_layout()
plt.show()

print(f"\nBreak-even GPU price: ~${cross_price:.2f}/GPU-hr")
print(f"Current rental price: ${cfg.rental.gpu_cost_per_hour}/GPU-hr")
print(f"→ At current price, {'renting' if m.total*12 < avg_annual_own else 'owning'} "
      f"is cheaper by ${abs(m.total*12 - avg_annual_own)/1e6:.1f}M/yr")

## 7. Ownership TCO Over Time vs Flat Rental

The ownership model has lumpy CapEx (hardware refresh) while rental is steady monthly.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

# Ownership: cumulative TCO over years
own_cumulative = np.cumsum([
    own_df.loc[i, 'total']
    for i in range(len(own_df))
])
ax.plot(own_df['year'], own_cumulative / 1e9,
        label='DC Ownership (cumulative)', color='#2196F3',
        linewidth=2, marker='o', markersize=4)

# Rental: cumulative (flat monthly × 12 per year)
rental_annual = m.total * MONTHS_PER_YEAR
rental_cumulative = np.cumsum([rental_annual] * len(own_years))
ax.plot(own_df['year'], rental_cumulative / 1e9,
        label='Cloud Rental (cumulative)', color='#4CAF50',
        linewidth=2, linestyle='--', marker='s', markersize=4)

ax.set_xlabel('Year')
ax.set_ylabel('Cumulative TCO ($B)')
ax.set_title('Cumulative TCO: Ownership vs Rental Over Time')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.1f}B'))
ax.legend()
plt.tight_layout()
plt.show()

## 8. Contract Term Sensitivity

How does the rental cost per GPU-hr change with contract length (longer = more setup amortization)?

In [ ]:
contract_months = [1, 3, 6, 12, 24, 36]
eff_costs = []

for cm in contract_months:
    c = copy.deepcopy(cfg)
    c.rental.contract_months = cm
    result = compute_rental_tco(c)
    eff_costs.append(result.cost_per_gpu_hour_effective)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([str(m) for m in contract_months], eff_costs, color='#9C27B0', alpha=0.8)
ax.axhline(own_eff_per_hr, color='#2196F3', linewidth=2, linestyle='--',
           label=f'DC Ownership equiv. ${own_eff_per_hr:.2f}/hr')
ax.set_xlabel('Contract Length (months)')
ax.set_ylabel('Effective $/GPU-hr')
ax.set_title('Rental Effective Cost vs Contract Term')
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:.2f}'))
ax.legend()

for i, v in enumerate(eff_costs):
    ax.text(i, v + 0.02, f'${v:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 9. Key Differences Summary

| Aspect | DC Ownership | Cloud Rental |
|--------|-------------|-------------|
| **Cost model** | CapEx-driven lifecycle simulation (15yr) | OpEx-only monthly calculator |
| **Hardware** | Multi-generation fleet (H100→B200→future) | Single GPU type, fixed cluster |
| **Demand** | Dynamic (grows with model size trends) | Static (fixed GPU count) |
| **Refresh** | Amortized server CapEx + decommission cycles | Implicit in provider pricing |
| **Goodput** | Single utilization factor | 3 explicit formulas (cold/hot/FT) |
| **Network** | Detailed topology (Ethernet/IB/OCS) + CapEx+OpEx | Flat monthly + egress fees |
| **Power** | PUE-based (energy + cooling provisioning) | Bundled in GPU $/hr |
| **Use case** | Cloud provider planning DC investments | Customer comparing cloud options |

### When to use which?
- **Ownership model**: You're a cloud provider or hyperscaler planning a new DC build. You need to model hardware refresh, power provisioning, and long-term demand growth.
- **Rental model**: You're a customer evaluating cloud GPU pricing. You want to compare providers, understand hidden costs (goodput, debugging), and find break-even vs building your own.